In [2]:
# run_demo_upsample.py

import torch
import torch.nn as nn
from transformers import AutoTokenizer

from phoneme_tokenizer import PhonemeTokenizer
from dataloader import build_dataloader
from model import MultiTaskModel


# ============================================================
# 1. Dataset contoh (pakai key 'input_ids' seperti punyamu)
# ============================================================

dataset_raw = [
    {
        "phonemes": ["həlˈoʊ", "dˈua", "pˈuluh"],
        "input_ids": [[15339], [1072, 64], [79, 360, 12825]]
    },
    {
        "phonemes": ["sˈatu", "dˈua"],
        "input_ids": [[4500], [1072, 64]]
    }
]

# Kita adapt jadi format yang dipakai dataloader_ctc (bpe_ids)
dataset = []
for item in dataset_raw:
    dataset.append({
        "phonemes": item["phonemes"],
        "input_ids": item["input_ids"]
    })

print("=== Dataset Contoh ===")
for d in dataset:
    print(d)

# ============================================================
# 2. Phoneme tokenizer
# ============================================================
ph_tok = PhonemeTokenizer.build_from_dataset(dataset)
print("\nPhoneme vocab size:", ph_tok.vocab_size)
print("Vocab:", ph_tok.stoi)

# ============================================================
# 3. BPE tokenizer (Llama)
# ============================================================
print("\nLoad BPE tokenizer...")
bpe_tok = AutoTokenizer.from_pretrained("GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct")
print("BPE vocab size:", bpe_tok.vocab_size)

# ============================================================
# 4. DataLoader CTC
# ============================================================
loader = build_dataloader(
    dataset=dataset,
    phoneme_tokenizer=ph_tok,
    bpe_tokenizer=bpe_tok,
    batch_size=2,
    shuffle=False
)

batch = next(iter(loader))
print("\n=== Batch dari Dataloader ===")
for k, v in batch.items():
    print(k, ":", v.shape if torch.is_tensor(v) else v)

X_ph       = batch["X_ph"]            # (B, T_ph)
Y_mlm      = batch["Y_mlm"]           # (B, T_ph)
Y_p2g      = batch["Y_p2g"]           # (B, T_bpe)
input_len  = batch["input_lengths"]   # (B,)
target_len = batch["target_lengths"]  # (B,)

# ============================================================
# 5. Bangun model upsample
# ============================================================
albert_cfg = {
    "vocab_size": ph_tok.vocab_size,
    "hidden_size": 128,
    "num_hidden_layers": 2,
    "num_attention_heads": 4,
    "intermediate_size": 256,
}

upsample_factor = 4  # bebas, tapi harus cukup besar supaya T_up >= T_bpe

print("\nBangun model (upsample_factor = %d) ..." % upsample_factor)
model = MultiTaskModel.from_albert_config(
    phoneme_vocab_size=ph_tok.vocab_size,
    bpe_vocab_size=bpe_tok.vocab_size,
    albert_config_dict=albert_cfg,
    pad_token_id=ph_tok.pad_token_id,
    blank_id=0,
    upsample_factor=upsample_factor,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

X_ph = X_ph.to(device)
Y_mlm = Y_mlm.to(device)
Y_p2g = Y_p2g.to(device)

# ============================================================
# 6. Forward
# ============================================================

logits_mlm, logits_ctc, extra = model(X_ph)

print("\nMLM logits shape:", logits_mlm.shape)    # (B, T_ph, V_ph)
print("CTC logits shape:", logits_ctc.shape)      # (B, T_up, V_bpe+1)

# hitung input_lengths untuk CTC (sudah di-upsample)
ctc_input_len = model.get_ctc_input_lengths(input_len)
print("input_lengths (phoneme):", input_len)
print("input_lengths (CTC):    ", ctc_input_len)
print("target_lengths:         ", target_len)

# ============================================================
# 7. Loss
# ============================================================

# --- MLM ---
mlm_logits = logits_mlm.reshape(-1, ph_tok.vocab_size)
mlm_targets = Y_mlm.reshape(-1)

mlm_loss_fn = nn.CrossEntropyLoss(ignore_index=ph_tok.pad_token_id)
loss_mlm = mlm_loss_fn(mlm_logits, mlm_targets)

# --- CTC ---
ctc_logits = logits_ctc.log_softmax(dim=-1).permute(1, 0, 2)  # (T_up, B, V)
ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

loss_ctc = ctc_loss_fn(
    ctc_logits,
    Y_p2g,
    ctc_input_len,
    target_len
)

loss_total = loss_mlm + loss_ctc

print("\nMLM Loss:", float(loss_mlm))
print("CTC Loss:", float(loss_ctc))
print("Total Loss:", float(loss_total))

# ============================================================
# 8. Backward (demo)
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer.zero_grad()
loss_total.backward()
optimizer.step()

print("\nUpdate sukses. Training dengan upsampling berjalan normal.")


=== Dataset Contoh ===
{'phonemes': ['həlˈoʊ', 'dˈua', 'pˈuluh'], 'input_ids': [[15339], [1072, 64], [79, 360, 12825]]}
{'phonemes': ['sˈatu', 'dˈua'], 'input_ids': [[4500], [1072, 64]]}

Phoneme vocab size: 7
Vocab: {'həlˈoʊ': 0, 'dˈua': 1, 'pˈuluh': 2, 'sˈatu': 3, '[PAD]': 4, '[MASK]': 5, '[UNK]': 6}

Load BPE tokenizer...
BPE vocab size: 128000

=== Batch dari Dataloader ===
X_ph : torch.Size([2, 3])
Y_mlm : torch.Size([2, 3])
Y_p2g : torch.Size([2, 6])
input_lengths : torch.Size([2])
target_lengths : torch.Size([2])

Bangun model (upsample_factor = 4) ...

MLM logits shape: torch.Size([2, 3, 7])
CTC logits shape: torch.Size([2, 12, 128001])
input_lengths (phoneme): tensor([3, 2])
input_lengths (CTC):     tensor([12,  8])
target_lengths:          tensor([6, 3])

MLM Loss: 2.2833266258239746
CTC Loss: 25.572284698486328
Total Loss: 27.85561180114746

Update sukses. Training dengan upsampling berjalan normal.


/tmp/ipykernel_3795/1215361176.py:144: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("\nMLM Loss:", float(loss_mlm))
